### Training a Random Forest Regressor model
Training a random forest regressor that predicts FPL points for the upcoming GW for players. This is a general model, and uses position as one of the predictors. A future step might be to produce a separate model for each position so that position-specific features can be better considered.

Rolling game statistics are key to the model - they will be computed on the previous three games for each player, and used as predictor features.

In [1]:
import os
import torch
import pandas as pd
import numpy as np
from model import AdvancedLSTM
import pickle
from eval import get_season_performance, season_performance_with_unlimited_transfers

In [2]:
base_path = os.getcwd()
base_path

'/Users/bragehs/Documents/FPL_forecast/predictor'

In [3]:
data_path = os.path.join(base_path, 'processed_data')
data_path

'/Users/bragehs/Documents/FPL_forecast/predictor/processed_data'

In [4]:
X_train = torch.load(data_path + '/X_train.pt', weights_only=True)
y_train = torch.load(data_path + '/y_train.pt', weights_only=True)
train_mapping = pd.read_csv(data_path + '/train_mapping.csv')

X_val = torch.load(data_path + '/X_val.pt', weights_only=True)
y_val = torch.load(data_path + '/y_val.pt', weights_only=True)

X_test = torch.load(data_path + '/X_test.pt', weights_only=True)
y_test = torch.load(data_path + '/y_test.pt', weights_only=True)
test_mapping = pd.read_csv(data_path + '/test_mapping.csv')

In [5]:
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")

X_train shape: torch.Size([185227, 5, 50])
y_train shape: torch.Size([185227, 1])
X_val shape: torch.Size([11384, 5, 50])
y_val shape: torch.Size([11384, 1])


In [6]:
# Find all test_mapping rows for Mohamed Salah
salah_mapping = test_mapping[test_mapping['name'] == 'mohamed_salah']

# Get the sequence indices for Salah
salah_sequence_indices = salah_mapping['sequence_idx'].values

# Extract the corresponding X_test data for Salah
salah_x_test = X_test[salah_sequence_indices]

# You can also get the corresponding y_test data
salah_y_test = y_test[salah_sequence_indices]

In [7]:
last_1_assists_idx = 2
last_1_goals_scored_idx = 7

In [8]:
#sanity check to see there is no data leakage
for i in range(salah_x_test.shape[0]):
    last_1_assists = salah_x_test[i, -1, last_1_assists_idx]
    last_1_goals_scored = salah_x_test[i, -1,  last_1_goals_scored_idx]
    
    print(f"Gameweek {i + 1}: Last 1 Assists: {last_1_assists}, Last 1 Goals Scored: {last_1_goals_scored}, Total Points: {salah_y_test[i]}")

Gameweek 1: Last 1 Assists: 0.0, Last 1 Goals Scored: 0.0, Total Points: tensor([14.])
Gameweek 2: Last 1 Assists: 0.25, Last 1 Goals Scored: 0.25, Total Points: tensor([10.])
Gameweek 3: Last 1 Assists: 0.0, Last 1 Goals Scored: 0.25, Total Points: tensor([17.])
Gameweek 4: Last 1 Assists: 0.5, Last 1 Goals Scored: 0.25, Total Points: tensor([2.])
Gameweek 5: Last 1 Assists: 0.0, Last 1 Goals Scored: 0.0, Total Points: tensor([6.])
Gameweek 6: Last 1 Assists: 0.25, Last 1 Goals Scored: 0.0, Total Points: tensor([10.])
Gameweek 7: Last 1 Assists: 0.0, Last 1 Goals Scored: 0.25, Total Points: tensor([3.])
Gameweek 8: Last 1 Assists: 0.0, Last 1 Goals Scored: 0.0, Total Points: tensor([12.])
Gameweek 9: Last 1 Assists: 0.25, Last 1 Goals Scored: 0.25, Total Points: tensor([10.])
Gameweek 10: Last 1 Assists: 0.0, Last 1 Goals Scored: 0.25, Total Points: tensor([9.])
Gameweek 11: Last 1 Assists: 0.0, Last 1 Goals Scored: 0.25, Total Points: tensor([14.])
Gameweek 12: Last 1 Assists: 0.25, 

In [10]:
best_model_data = torch.load("best_model.pth", map_location=torch.device('cpu'))

/var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/ipykernel_22626/2890380057.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  best_model_data = torch.load("best_model.pth"

In [11]:
print(best_model_data['hidden_dim'])
print(best_model_data['num_layers'])
print(best_model_data['num_fc_layers'])

64
1
3


In [12]:
model = AdvancedLSTM(
    hidden_dim=best_model_data['hidden_dim'],
    num_layers=best_model_data['num_layers'],
    input_dim= best_model_data['input_dim'],
    output_dim=1,
    num_fc_layers=best_model_data['num_fc_layers'],
) 
model.load_state_dict(best_model_data['model_state_dict'])

/opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  warnings.warn(


<All keys matched successfully>

In [13]:
#print number of parameters in the model
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of parameters in the model: {num_params}")

Number of parameters in the model: 32417


In [14]:
model.eval()

AdvancedLSTM(
  (lstm): LSTM(50, 64, batch_first=True, dropout=0.3)
  (fc_layers): ModuleList(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): Linear(in_features=32, out_features=16, bias=True)
    (2): Linear(in_features=16, out_features=1, bias=True)
  )
  (batch_norms): ModuleList(
    (0): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (relu): ReLU()
)

In [15]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

torch.Size([185227, 5, 50])
torch.Size([11384, 5, 50])
torch.Size([11567, 5, 50])


In [16]:
test = pd.read_csv(data_path + '/test_data.csv')

In [17]:
predictions = model(X_test).detach().numpy()
print(predictions.shape)
print(y_test.shape)

(11567, 1)
torch.Size([11567, 1])


In [18]:
print(torch.mean(y_test))
print(torch.var(y_test))    

tensor(2.7053)
tensor(8.3782)


In [19]:
print(np.mean(predictions))
print(np.var(predictions))

2.6885464
0.72525454


In [20]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

rmse = root_mean_squared_error(y_test, predictions)
print(f"RMSE: {rmse}")

mae = mean_absolute_error(y_test, predictions)
print(f"MAE: {mae}")


RMSE: 2.781411647796631
MAE: 1.9658403396606445


In [21]:
X_test.shape

torch.Size([11567, 5, 50])

In [22]:
salah_y_test[0]

tensor([14.])

In [23]:
remaining_lagged_features = sorted(pickle.load(open(data_path + '/remaining_lagged_features.pkl', 'rb')))
remaining_lagged_features

['lagged_fixture_difficulty',
 'lagged_was_home',
 'last_1_assists',
 'last_1_bonus',
 'last_1_clean_sheets',
 'last_1_creativity',
 'last_1_goals_conceded',
 'last_1_goals_scored',
 'last_1_ict_index',
 'last_1_influence',
 'last_1_minutes',
 'last_1_red_cards',
 'last_1_result_encoded',
 'last_1_threat',
 'last_1_yellow_cards',
 'last_3_assists',
 'last_3_bonus',
 'last_3_clean_sheets',
 'last_3_creativity',
 'last_3_goals_conceded',
 'last_3_goals_scored',
 'last_3_ict_index',
 'last_3_influence',
 'last_3_minutes',
 'last_3_red_cards',
 'last_3_result_encoded',
 'last_3_threat',
 'last_3_yellow_cards',
 'last_5_assists',
 'last_5_bonus',
 'last_5_clean_sheets',
 'last_5_goals_conceded',
 'last_5_goals_scored',
 'last_5_influence',
 'last_5_red_cards',
 'last_5_result_encoded',
 'last_5_yellow_cards',
 'last_all_assists',
 'last_all_bonus',
 'last_all_clean_sheets',
 'last_all_creativity',
 'last_all_goals_conceded',
 'last_all_goals_scored',
 'last_all_ict_index',
 'last_all_influe

In [24]:
for i in range(0, len(remaining_lagged_features)):
    if salah_x_test[0, -1, i] != 0:
        print(f"Feature {remaining_lagged_features[i]} is not zero")
        print("Value is ", salah_x_test[0, -1, i])
        print()

Feature lagged_fixture_difficulty is not zero
Value is  tensor(0.4000)

Feature position_encoded is not zero
Value is  tensor(2.)



In [25]:
salah_y_test[16]

tensor([21.])

In [26]:
test = pd.read_csv(data_path + '/test_data.csv')
salah_df = test[test['name'] == 'mohamed_salah']
salah_df[['name', 'GW', 'total_points', 'last_1_assists', 'last_1_goals_scored', 'last_1_ict_index']]

,name,GW,total_points,last_1_assists,last_1_goals_scored,last_1_ict_index
108,mohamed_salah,1.0,14.0,0.00,0.00,0.000000
420,mohamed_salah,2.0,10.0,0.25,0.25,0.424581
746,mohamed_salah,3.0,17.0,0.00,0.25,0.270950
1063,mohamed_salah,4.0,2.0,0.50,0.25,0.449721
1372,mohamed_salah,5.0,6.0,0.00,0.00,0.231844
1690,mohamed_salah,6.0,10.0,0.25,0.00,0.558659
1994,mohamed_salah,7.0,3.0,0.00,0.25,0.354749
2305,mohamed_salah,8.0,12.0,0.00,0.00,0.117318
2613,mohamed_salah,9.0,10.0,0.25,0.25,0.321229
2924,mohamed_salah,10.0,9.0,0.00,0.25,0.282123


In [33]:
for i in range(0, len(salah_x_test)):
    input_to_game = salah_x_test[i].unsqueeze(0)
    print(f"Gameweek {i + 1}:")
    print("prediction", model(input_to_game).detach().numpy())
    print("actual", salah_y_test[i].item())

Gameweek 1:
prediction [[1.7004305]]
actual 14.0
Gameweek 2:
prediction [[3.6332295]]
actual 10.0
Gameweek 3:
prediction [[3.2940903]]
actual 17.0
Gameweek 4:
prediction [[4.2217674]]
actual 2.0
Gameweek 5:
prediction [[4.6880326]]
actual 6.0
Gameweek 6:
prediction [[5.8495173]]
actual 10.0
Gameweek 7:
prediction [[5.89323]]
actual 3.0
Gameweek 8:
prediction [[6.3016515]]
actual 12.0
Gameweek 9:
prediction [[5.7335]]
actual 10.0
Gameweek 10:
prediction [[6.1430726]]
actual 9.0
Gameweek 11:
prediction [[5.4677267]]
actual 14.0
Gameweek 12:
prediction [[5.6366725]]
actual 13.0
Gameweek 13:
prediction [[6.3887467]]
actual 13.0
Gameweek 14:
prediction [[6.517064]]
actual 18.0
Gameweek 15:
prediction [[7.2635403]]
actual 13.0
Gameweek 16:
prediction [[7.409186]]
actual 5.0
Gameweek 17:
prediction [[6.685003]]
actual 21.0
Gameweek 18:
prediction [[7.4882407]]
actual 9.0
Gameweek 19:
prediction [[6.9127555]]
actual 16.0
Gameweek 20:
prediction [[7.699634]]
actual 7.0
Gameweek 21:
prediction [

In [34]:
scores, total_score = get_season_performance(
    y_test=y_test,
    predictions=predictions,
    remaining_lagged_features=remaining_lagged_features
)

Players with NaN total_points_last_season: []
Number of NaN values remaining: 0
0.0 :  daniel_bentley
1.0 :  luke_thomas
2.0 :  matheus_franca_de_oliveira
3.0 :  daniel_jebbison
Bench players: ['daniel_bentley', 'luke_thomas', 'matheus_franca_de_oliveira', 'daniel_jebbison']
Bench cost: 167
length available players df 562
Available cash: 833
Decision variables: [aaron_cresswell, aaron_ramsdale, aaron_wan_bissaka, abdoulaye_doucoure, abdukodir_khusanov, abdul_fatawu, adam_armstrong, adam_lallana, adam_smith, adam_webster, adam_wharton, adama_traore, albert_gronbaek, alejandro_garnacho, alex_iwobi, alex_mccarthy, alex_moreno_lopera, alex_palmer, alex_scott, alexander_isak, alexis_mac_allister, alfie_dorrington, alfie_pond, ali_al_hamadi, alisson_ramses_becker, alphonse_areola, altay_bayindir, amad_diallo, amadou_onana, andre_onana, andre_trindade_da_costa_neto, andreas_hoelgebaum_pereira, andres_garcia, andrew_robertson, andy_irving, anthony_elanga, anthony_gordon, antoine_semenyo, anton

In [35]:
scores

,GW,player_in,player_out,total_score
0,1.0,,,33.0
1,2.0,danny_welbeck,jean_philippe_mateta,72.0
2,3.0,pedro_porro,william_saliba,33.0
3,4.0,luis_diaz,son_heung_min,47.0
4,5.0,mohamed_salah,phil_foden,59.0
5,6.0,matheus_santos_carneiro_da_cunha,danny_welbeck,80.0
6,7.0,,,60.0
7,8.0,ashley_young,joachim_andersen,41.0
8,9.0,danny_welbeck,matheus_santos_carneiro_da_cunha,68.0
9,10.0,rayan_ait_nouri,jarrad_branthwaite,38.0


In [36]:
total_score.item()

2055.0

In [37]:
scores, total_score = season_performance_with_unlimited_transfers(
    y_test=y_test,
    predictions=predictions,
    remaining_lagged_features=remaining_lagged_features
)

Players with NaN total_points_last_season: []
Number of NaN values remaining: 0
0.0 :  daniel_bentley
1.0 :  luke_thomas
2.0 :  matheus_franca_de_oliveira
3.0 :  daniel_jebbison
Bench players: ['daniel_bentley', 'luke_thomas', 'matheus_franca_de_oliveira', 'daniel_jebbison']
Bench cost: 167
Simulating season with unlimited transfers for 38 gameweeks
Available budget per gameweek: 833

--- Gameweek 1.0 ---
Players available for GW 1.0: 316
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/2bcbe6b0019b48ca9f01b605c5a78d31-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/2bcbe6b0019b48ca9f01b605c5a78d31-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 30 COLUMNS
At line 1927 RHS
At line 1953 BOUNDS
At

In [38]:
total_score.item()

2356.0

In [39]:
scores

,team,gw_score
0,"[adam_armstrong, cameron_archer, daniel_castel...",36.0
1,"[antoine_semenyo, bukayo_saka, danny_welbeck, ...",62.0
2,"[andrew_robertson, bukayo_saka, cole_palmer, c...",66.0
3,"[andrew_robertson, bukayo_saka, cole_palmer, d...",31.0
4,"[andrew_robertson, arijanet_muric, bukayo_saka...",82.0
5,"[andre_onana, andrew_robertson, dwight_mcneil,...",66.0
6,"[bukayo_saka, cole_palmer, dwight_mcneil, erli...",65.0
7,"[ashley_young, cole_palmer, dwight_mcneil, erl...",56.0
8,"[bukayo_saka, cole_palmer, david_raya_martin, ...",66.0
9,"[bryan_mbeumo, bukayo_saka, cole_palmer, david...",48.0
